# Notebook 16: Robustness table and figure inventory

**Goal.** Show that the main finding survives reasonable alternative choices (the 'multiverse'), and list every figure and table to deposit.

**Cell 1.** Loads the tools and the shared settings. **sys.path.insert** tells Python where the project code lives; **mlc.config** gives every notebook the same folders (PROC = processed data, RES = results tables, FIG = figures) and the profile (*sample* = built-in simulated data, *real* = your downloads). It also tells pandas to print wide tables on one screen. This cell is identical in every notebook.

In [ ]:
import sys; sys.path.insert(0, "../src")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from mlc.config import *
from mlc import genesets as G
pd.set_option("display.width", 150); pd.set_option("display.max_columns", 20)
print("profile:", PROFILE, "| cohorts:", cohorts())

**Cell 2.** Repeats the pooled age-slope under five analysis variants (no stress adjustment, stricter minimum cell count, DAM cells removed, no covariates...). **What reviewers want to see:** the same sign and a similar size in every row. If one variant flips the answer, report it and explain.

*You should see:* Five rows, all about -0.07 with intervals below 0.

In [ ]:
from mlc.dataset import Dataset
from mlc.coupling import *
from mlc import plots
sheet = pd.read_csv(SHEET).set_index("sample_id")
obsd = {c: Dataset.load(PROC / f"{c}_scored").obs for c in cohorts()["mouse"]}
variants = {"primary (M2)": dict(cov=["depth", "pct_mt", "dissoc", "STATE"], min_cells=100, drop=None),
            "no stress covariate": dict(cov=["depth", "pct_mt", "STATE"], min_cells=100, drop=None),
            "min 200 cells": dict(cov=["depth", "pct_mt", "dissoc", "STATE"], min_cells=200, drop=None),
            "without DAM cells": dict(cov=["depth", "pct_mt", "dissoc"], min_cells=60, drop="DAM"),
            "no covariates (M0)": dict(cov=[], min_cells=100, drop=None)}
rows = []
for name, v in variants.items():
    parts = []
    for c, o in obsd.items():
        o = o if v["drop"] is None else o[o.state != v["drop"]]
        sc_ = [x for x in o.columns if x.startswith("state_")] if "STATE" in v["cov"] else []
        r = coupling_per_sample(o, "m_mitophagy", "m_lysosome", covars=[x for x in v["cov"] if x != "STATE"] + sc_,
                                min_cells=v["min_cells"], n_boot=0); r["cohort"], r["level"] = c, "x"; parts.append(r)
    Rv = pd.concat(parts); Rv["log2_age"] = np.log2(Rv.sample_id.map(sheet.age_months).astype(float))
    m = dersimonian_laird(*cohort_slopes(Rv)[["beta", "se"]].values.T); rows.append(dict(variant=name, **m))
rob = pd.DataFrame(rows); rob.to_csv(RES / "robustness.csv", index=False)
print(rob[["variant", "beta", "ci_lo", "ci_hi", "p"]].round(3))

**Cell 3.** Forest plot of the five variants.

In [ ]:
plots.fig_robust(rob); plt.show()

**Cell 4.** Lists every figure and results table. Open the PNGs in **Inkscape** to assemble multi-panel journal figures (export as SVG/PDF; Arial or Helvetica at 5-7 pt). Deposit the tables with the paper.

In [ ]:
inv = pd.DataFrame({"file": sorted(p.name for p in FIG.glob("*.png"))})
inv.to_csv(RES / "figure_inventory.csv", index=False)
print("figures:"); print(inv.to_string(index=False))
print("\nresult tables:", sorted(p.name for p in RES.glob("*.csv"))[:30])